In [2]:
# ECE formulation
def compute_ece(y_true,y_prob, n_bins=10):
    y_true = np.asarray(y_true)
    predictions = np.argmax(y_prob, axis=1)
    confidences = np.max(y_prob, axis = 1) 
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0
    n = len(y_true)
    for i in range(n_bins):
        start, end = bin_edges[i], bin_edges[i + 1]
        bin_mask = (confidences > start) & (confidences <= end)
        bin_size = np.sum(bin_mask)
        if (bin_size > 0):
            avg_conf = np.mean(confidences[bin_mask])
            avg_acc = np.mean(predictions[bin_mask] == y_true[bin_mask])
            ece += (bin_size / n) * np.abs(avg_conf - avg_acc) 
    return ece 


# have to compute all permutations to take the lowest ECE
def preprocess_ece(y_true, y_probs, batch_class):
    best_ece = np.inf 

    # If the model outputs fewer classes then expected, we need to pad. This ensures probability arrays always have batch_class columns.
    if y_probs.shape[1] < batch_class:
        y_probs = np.hstack([y_probs, np.zeros((y_probs.shape[0], batch_class - y_probs.shape[1]))])
        
    all_perms =list(itertools.permutations(range(max(batch_class, y_probs.shape[1])),batch_class))

    for permutation in all_perms:
        curr_ece = compute_ece(y_true, y_probs[:, permutation]) 
        if curr_ece < best_ece:
            best_ece = curr_ece
    return best_ece

In [3]:
import numpy as np
import random
from scipy.stats import invwishart, chi2

def generate_t_mixture_data(weight_concentration_prior=1,mean_precision_prior=0.1,features=2,mean_prior=0.0,
                            degrees_of_freedom_prior=5,nu_prior = 2, 
                            covariance_prior=None, seed = 42):
    if covariance_prior is None:
        covariance_prior = np.eye(features)

    np.random.seed(seed)
    random.seed(seed)
    n_components = np.random.randint(2, 8)
    seq_len = np.random.randint(100, 500)

    # Step 1: sample mixture weights
    weights = np.random.dirichlet(np.ones(n_components) * weight_concentration_prior)
    counts = np.random.multinomial(seq_len, weights)

    # Step 2: sample component parameters from priors
    means = []
    covariances = []
    for _ in range(n_components):
        Sigma = invwishart.rvs(df=degrees_of_freedom_prior, scale=covariance_prior)
        mu = np.random.multivariate_normal(np.full(features, mean_prior), Sigma / mean_precision_prior)
        means.append(mu)
        covariances.append(Sigma)

    # Step 3: sample data from Student-t components
    X = []
    y = []
    for k in range(n_components):
        n_k = counts[k]
        mu_k = means[k]
        Sigma_k = covariances[k]
        nu = nu_prior

        # Sample from multivariate t-distribution using scale mixture representation
        w = chi2.rvs(df=nu, size=n_k) / nu  # scaling variable
        z = np.random.multivariate_normal(np.zeros(features), Sigma_k, size=n_k)
        X_k = mu_k + z / np.sqrt(w)[:, None]

        X.extend(X_k)
        y.extend(np.full(n_k, k))

    return np.array(X), np.array(y), n_components